In [1]:
from libTools import *
from uvars import tdem_fn,edem_fn,them_fn,tgap_fn,esan_fn,esar_fn,egmg_fn,sen2_fn,sen1_fn

In [2]:
from os.path import basename, join, dirname

In [3]:
tif = sen2_fn
fdir = dirname(tif)
fname = basename(tif).replace('.tif', '')
split_raster_stack_to_bands(stack_path=sen2_fn, 
                            outdir=fdir,
                            basename=fname, 
                            overwrite=False)

Saved: /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_s2_B1.tif
Saved: /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_s2_B2.tif
Saved: /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_s2_B3.tif


In [4]:
tif = sen1_fn
fdir = dirname(tif)
fname = basename(tif).replace('.tif', '')
split_raster_stack_to_bands(stack_path=tif, 
                            outdir=fdir,
                            basename=fname, 
                            overwrite=False)

Saved: /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_s1_B1.tif
Saved: /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_s1_B2.tif


In [6]:
import os
import json
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import Point

def load_raster_to_geodf(input_path, outdir, overwrite=False):
    """
    Load a raster, clean invalid values, extract valid (lon, lat, elev) points,
    write them to a GeoPackage and the profile as JSON.

    Parameters:
    - input_path (str): Path to the input raster (.tif)
    - outdir (str): Directory to save outputs (.gpkg and .json)
    - overwrite (bool): Whether to overwrite existing outputs (default: False)
    """
    os.makedirs(outdir, exist_ok=True)

    basename = os.path.splitext(os.path.basename(input_path))[0]
    gpkg_path = os.path.join(outdir, f"{basename}.gpkg")
    json_path = os.path.join(outdir, f"{basename}.json")

    if not overwrite and os.path.exists(gpkg_path) and os.path.exists(json_path):
        print(f"Skipped: {basename} outputs already exist.")
        return gpkg_path, json_path

    with rasterio.open(input_path) as src:
        data = src.read(1).astype(np.float32)
        profile = src.profile
        transform = src.transform
        nodata_val = src.nodata

    # Mask invalid values
    if nodata_val is not None:
        data[data == nodata_val] = np.nan
    data[data < -99] = np.nan
    data[data > 1000] = np.nan

    # Get row, col of valid pixels
    rows, cols = np.where(~np.isnan(data))
    zs = data[rows, cols]

    # Convert to coordinates
    xs, ys = rasterio.transform.xy(transform, rows, cols)
    points = [Point(x, y) for x, y in zip(xs, ys)]

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame({'elevation': zs}, geometry=points, crs=profile['crs'])

    # Save GeoPackage
    gdf.to_file(gpkg_path, driver='GPKG')

    # Save profile as JSON
    with open(json_path, 'w') as f:
        json.dump(profile, f, indent=2, default=str)

    print(f"Saved: {gpkg_path}")
    print(f"Saved: {json_path}")
    return gpkg_path, json_path


In [11]:
import os
import json
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm

def tif2vec_chunked(input_path, outdir, overwrite=False):
    """
    Loads a raster in chunks, filters invalid values, extracts valid (lon, lat, elev),
    saves to GPKG, SHP, and Parquet, and dumps profile to JSON.

    Args:
        input_path (str): Path to input .tif raster
        outdir (str): Output directory for files
        overwrite (bool): Whether to overwrite existing outputs
    """
    os.makedirs(outdir, exist_ok=True)
    basename = os.path.splitext(os.path.basename(input_path))[0]

    gpkg_path = os.path.join(outdir, f"{basename}.gpkg")
    shp_path = os.path.join(outdir, f"{basename}.shp")
    parquet_path = os.path.join(outdir, f"{basename}.parquet")
    json_path = os.path.join(outdir, f"{basename}.json")

    # Check if all outputs exist
    if (not overwrite and 
        all(os.path.exists(p) for p in [gpkg_path, shp_path, parquet_path, json_path])):
        print(f"Skipped: Outputs for {basename} already exist.")
        return gpkg_path, shp_path, parquet_path, json_path

    geometries = []
    elevations = []

    with rasterio.open(input_path) as src:
        profile = src.profile
        nodata_val = src.nodata
        crs = src.crs

        # Loop through raster blocks
        for _, window in tqdm(src.block_windows(1), desc="Processing blocks"):
            data = src.read(1, window=window).astype(np.float32)
            row_off, col_off = window.row_off, window.col_off

            # Build row/col index arrays
            rows, cols = np.where(~np.isnan(data) & (data > -99) & (data < 1000))
            if nodata_val is not None:
                rows, cols = rows[data[rows, cols] != nodata_val], cols[data[rows, cols] != nodata_val]

            if len(rows) == 0:
                continue

            # Elevation values
            zs = data[rows, cols]

            # Convert to coordinates
            xs, ys = rasterio.transform.xy(src.transform, rows + row_off, cols + col_off)
            geometries.extend([Point(x, y) for x, y in zip(xs, ys)])
            elevations.extend(zs.tolist())

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame({'elevation': elevations}, geometry=geometries, crs=crs)

    # Save files
    gdf.to_file(gpkg_path, driver="GPKG")
    gdf.to_file(shp_path, driver="ESRI Shapefile")
    gdf.to_parquet(parquet_path, index=False)

    # Save profile
    with open(json_path, "w") as f:
        json.dump(profile, f, indent=2, default=str)

    print(f"Saved: {gpkg_path}, {shp_path}, {parquet_path}, {json_path}")
    return gpkg_path, shp_path, parquet_path, json_path


In [8]:
outdir = 'tmp/tif2vec'

In [ ]:
load_raster_to_geodf(input_path=tgap_fn, outdir=outdir, overwrite=False)
#tif2vec_chunked(input_path=tgap_fn, outdir=outdir, overwrite=False)

Skipped: N10E105_tdem_dem_egm_v_3_gap outputs already exist.


Processing blocks: 0it [00:00, ?it/s]

Processing blocks: 2056it [00:43, 47.56it/s]


In [4]:
import os
import json
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm
from rasterio.windows import Window

def load_raster_to_geodf_chunked(input_path, outdir, overwrite=False, ps=256):
    """
    Loads a raster in user-defined chunks, filters invalid values,
    extracts valid (lon, lat, elev) points, and writes to multiple formats.

    Args:
        input_path (str): Path to input .tif raster
        outdir (str): Output directory
        overwrite (bool): Whether to overwrite existing outputs
        ps (int): Patch size (default 256); window will be ps x ps
    """
    os.makedirs(outdir, exist_ok=True)
    basename = os.path.splitext(os.path.basename(input_path))[0]

    gpkg_path = os.path.join(outdir, f"{basename}.gpkg")
    shp_path = os.path.join(outdir, f"{basename}.shp")
    parquet_path = os.path.join(outdir, f"{basename}.parquet")
    json_path = os.path.join(outdir, f"{basename}.json")

    if (not overwrite and 
        all(os.path.exists(p) for p in [gpkg_path, shp_path, parquet_path, json_path])):
        print(f"Skipped: Outputs for {basename} already exist.")
        return gpkg_path, shp_path, parquet_path, json_path

    geometries = []
    elevations = []

    with rasterio.open(input_path) as src:
        profile = src.profile
        nodata_val = src.nodata
        crs = src.crs
        width, height = src.width, src.height

        # Iterate over custom tile windows
        n_cols = (width + ps - 1) // ps
        n_rows = (height + ps - 1) // ps

        for row in tqdm(range(n_rows), desc="Processing rows"):
            for col in range(n_cols):
                row_off = row * ps
                col_off = col * ps
                win = Window(col_off=col_off, row_off=row_off,
                             width=min(ps, width - col_off),
                             height=min(ps, height - row_off))
                data = src.read(1, window=win).astype(np.float32)

                # Masking and filtering
                mask = (~np.isnan(data)) & (data > -99) & (data < 1000)
                if nodata_val is not None:
                    mask &= (data != nodata_val)

                rows, cols = np.where(mask)
                if rows.size == 0:
                    continue

                zs = data[rows, cols]
                xs, ys = rasterio.transform.xy(src.window_transform(win), rows, cols)
                geometries.extend([Point(x, y) for x, y in zip(xs, ys)])
                elevations.extend(zs.tolist())

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame({'elevation': elevations}, geometry=geometries, crs=crs)

    # Write outputs
    gdf.to_file(gpkg_path, driver="GPKG")
    gdf.to_file(shp_path, driver="ESRI Shapefile")
    gdf.to_parquet(parquet_path, index=False)

    with open(json_path, "w") as f:
        json.dump(profile, f, indent=2, default=str)

    print(f"Saved: {gpkg_path}, {shp_path}, {parquet_path}, {json_path}")
    return gpkg_path, shp_path, parquet_path, json_path


In [5]:
outdir = 'tmp/tif2vec'
from uvars import tgap_fn

In [6]:
load_raster_to_geodf_chunked(
    input_path=tgap_fn,
    outdir=outdir,
    overwrite=True,
    ps=512  # Use 512x512 tiles
)


Processing rows: 100%|██████████| 5/5 [00:38<00:00,  7.72s/it]


Saved: tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.gpkg, tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.shp, tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.parquet, tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.json


('tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.gpkg',
 'tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.shp',
 'tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.parquet',
 'tmp/tif2vec/N10E105_tdem_dem_egm_v_3_gap.json')